# Lab 00: Environment Setup

**Goal:** get a working Python environment, choose a model provider, and make your first Strands call.

Based on [sample-building-with-strands-course](https://github.com/aws-samples/sample-building-with-strands-course) (`README.md` + `SETUP.md`), tested against `strands-agents==1.59.0`.

## What is Strands?
[Strands Agents](https://strandsagents.com) is an open-source, model-agnostic SDK from AWS for building **agent harnesses**: the loop that calls a model, runs tools, feeds results back, and repeats until the task is done.

## 1. Create the environment (terminal, one time)
Run these from the repo root (`strands-getting-started/`), **not** inside Jupyter:

```bash
# Python 3.10+ is required. 3.12 is the course's tested version.
uv venv --python 3.12 .venv          # or: python3.12 -m venv .venv
source .venv/bin/activate
uv pip install -r requirements.txt   # or: pip install -r requirements.txt

# Register the venv as a Jupyter kernel and launch
python -m ipykernel install --user --name strands-labs --display-name "Strands Labs (.venv)"
jupyter lab
```

Then open this notebook and select the **Strands Labs (.venv)** kernel.

## 2. Verify the install

In [ ]:
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 10), "Strands needs Python 3.10+"

for pkg in ["strands-agents", "mcp", "boto3"]:
    print(f"{pkg:16} {version(pkg)}")

import strands  # if this fails, you're on the wrong kernel
print("strands imported from", strands.__file__)

## 3. Choose a model provider

All labs get their model from `lab_config.get_model()`, configured by two variables:

| `LAB_PROVIDER` | Needs | Default `LAB_MODEL_ID` |
|---|---|---|
| `bedrock` (default) | AWS credentials + Bedrock access | Strands' built-in Bedrock default |
| `anthropic` | `ANTHROPIC_API_KEY` | `claude-sonnet-5-5` |
| `openai` | `OPENAI_API_KEY` | `gpt-4o` |
| `ollama` | [Ollama](https://ollama.com) running locally | `gemma4:latest` (needs a model that supports tool calling) |

Copy `.env.example` to `.env` at the repo root and set them there. Your shell environment takes precedence over `.env`.

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 4a. Bedrock: check AWS credentials
Skip this if you aren't using Bedrock.

If you don't have credentials yet:
```bash
brew install awscli
aws configure            # access key / secret / region us-east-1 / json
# or, with IAM Identity Center:
aws configure sso && aws sso login
```
`us-east-1` has the broadest model availability. Claude models on Bedrock submit a one-time use-case acknowledgment on first call. If the first call fails with an access error, wait a couple of minutes and try again.

In [ ]:
from lab_config import provider

if provider() == "bedrock":
    import boto3, os
    region = os.environ.get("AWS_REGION", "us-east-1")
    session = boto3.Session()
    print("Identity:", session.client("sts").get_caller_identity()["Arn"])

    bedrock = session.client("bedrock", region_name=region)
    models = bedrock.list_foundation_models()["modelSummaries"]
    anthropic = sorted(m["modelId"] for m in models if m["providerName"] == "Anthropic")
    print(f"{len(models)} foundation models in {region}; Anthropic examples:", anthropic[:5])
else:
    print("Not using Bedrock; skipping.")

## 4b. Ollama: check the local server
Skip this if you aren't using Ollama. Pull a tool-capable model first, for example `ollama pull gemma4:latest`.

In [ ]:
if provider() == "ollama":
    import json, urllib.request
    from lab_config import model_id
    with urllib.request.urlopen("http://localhost:11434/api/tags") as r:
        names = [m["name"] for m in json.load(r)["models"]]
    print("Local models:", names)
    assert model_id() in names, f"Run: ollama pull {model_id()}"
else:
    print("Not using Ollama; skipping.")

## 5. Your first agent
This is the course's `01-agent-loop/simple_agent.py`. By default the agent streams its answer to the output as it's generated.

In [ ]:
from strands import Agent

agent = Agent(model=get_model())
result = agent("Explain what an AI agent is in two sentences.")

If you see a two-sentence answer, you're set up. ✅

### Troubleshooting
- **`ModuleNotFoundError: strands`**: wrong kernel. Select *Strands Labs (.venv)*.
- **`NoCredentialsError` / `ExpiredToken` / SSO expired**: run `aws sso login` (or `aws configure`), then restart the kernel.
- **`ThrottlingException`**: free-tier Claude limits are low. Set `LAB_MODEL_ID=amazon.nova-pro-v1:0` or `us.amazon.nova-lite-v1:0`.
- **`AccessDeniedException` on the first Claude call**: the use-case form is still processing. Wait a few minutes.

**Next:** [Lab 01: The Agent Loop](01-agent-loop.ipynb)